# Lesson 22 Lab — Pruning Transformer Heads, FFN Neurons, and Layers

**Puzzle:** Which structural unit changes Transformer compute rather than only masking values?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Attention heads, FFN intermediate neurons, hidden dimensions, and whole layers are different dependency units. Masking a head preserves the packed QKV and output projection shapes, while physically reducing FFN width changes dense GEMMs. Whole-layer removal changes depth and residual composition. Each route needs its own quality and latency evidence.


## 0. Predict before running

1. Predict which candidate changes physical parameter count.
2. Estimate the relative attention and FFN work for the chosen S, D, and D_ff.
3. Predict which route has the largest output drift before recovery.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A compact pre-norm Transformer block exposes head outputs, an FFN intermediate, residuals, and a two-block stack. The lab compares a head mask, a half-width physical FFN, and layer skipping under one CUDA workload.

- A head mask is not automatically a narrower attention operator.
- FFN width directly controls two dense matrix multiplications.
- Layer pruning changes depth and residual transformations.


## 2. Derive the mechanism

For sequence length S and hidden width D, attention projections scale roughly with `S D²` and score/value work with `S² D`; FFN work scales with `S D D_ff`. Removing a logical head but retaining packed D-wide projections may leave most work unchanged. Halving D_ff directly reduces two GEMM dimensions. Removing a block deletes both attention and FFN work but creates a larger functional perturbation. Structural claims must specify which dimensions changed.

### Mechanism at a glance

```mermaid
flowchart TD
  T["Transformer block"] --> A["Attention heads"]
  T --> F["FFN neurons"]
  T --> L["whole-layer depth"]
  A --> QA["slice Q/K/V + output projection"]
  F --> QF["slice up/gate + down projection"]
  L --> QL["update layer list + cache/config"]
  QA --> V["rebuild, validate, benchmark"]
  QF --> V
  QL --> V
```

### Walk it step by step

1. **Choose the structural unit.** Attention heads, hidden channels, FFN neurons, and full layers change different dimensions and interfaces.
2. **Propagate coupled dimensions.** Head removal affects Q/K/V and output projection slices; FFN removal couples up and down projections.
3. **Rebuild the executable graph.** Config fields, cache shapes, residual dimensions, and exported metadata must agree with the new structure.
4. **Measure the remaining bottleneck.** A smaller attention block may not improve end-to-end latency when FFN, memory traffic, or launch overhead dominates.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 22
LESSON_TITLE = 'Pruning Transformer Heads, FFN Neurons, and Layers'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260830
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | full block/stack and same-shape attention-head masking |
| Candidate | physically narrowed FFN and one-layer-shorter stack |
| Held constant | weights where comparable, input, sequence length, batch, hidden width, dtype, eval mode, and timing |
| Measurements | physical parameters, output RMSE/cosine, median latency, and theoretical work components |
| Evidence | `pytorch-gpu` |

**Experiment:** Measure head masking, physical FFN narrowing, and whole-layer skipping in a compact CUDA Transformer.


## 5. Read the experiment code

The block returns a head-mask path without rewriting packed projections, making its unchanged physical shape visible. The FFN candidate copies selected intermediate rows/columns into smaller linear modules. Layer skipping reuses the first block output. These controls keep three pruning units conceptually separate.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
class TinyBlock(nn.Module):
    def __init__(self,d=64,heads=4,ff=256):
        super().__init__(); self.d=d; self.heads=heads; self.hd=d//heads; self.ln1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.proj=nn.Linear(d,d); self.ln2=nn.LayerNorm(d); self.fc1=nn.Linear(d,ff); self.fc2=nn.Linear(ff,d)
    def forward(self,x,head_mask=None):
        h=self.ln1(x); b,s,_=h.shape; qkv=self.qkv(h).view(b,s,3,self.heads,self.hd).permute(2,0,3,1,4); q,k,v=qkv
        attn=torch.softmax(q@k.transpose(-2,-1)/math.sqrt(self.hd),dim=-1); heads=attn@v
        if head_mask is not None: heads=heads*head_mask.view(1,-1,1,1)
        a=self.proj(heads.transpose(1,2).reshape(b,s,self.d)); x=x+a; return x+self.fc2(F.gelu(self.fc1(self.ln2(x))))
full=TinyBlock().to(DEVICE).to(torch.bfloat16).eval(); narrow=TinyBlock(ff=128).to(DEVICE).to(torch.bfloat16).eval()
with torch.no_grad():
    narrow.ln1.load_state_dict(full.ln1.state_dict()); narrow.qkv.load_state_dict(full.qkv.state_dict()); narrow.proj.load_state_dict(full.proj.state_dict()); narrow.ln2.load_state_dict(full.ln2.state_dict()); narrow.fc1.weight.copy_(full.fc1.weight[:128]); narrow.fc1.bias.copy_(full.fc1.bias[:128]); narrow.fc2.weight.copy_(full.fc2.weight[:,:128]); narrow.fc2.bias.copy_(full.fc2.bias)
x=torch.randn(8,128,64,device=DEVICE,dtype=torch.bfloat16); head_mask=torch.tensor([1,0,1,0],device=DEVICE,dtype=torch.bfloat16)
with torch.inference_mode(): ref=full(x); hm=full(x,head_mask); fn=narrow(x); skip=x
tf=timing_summary(cuda_times(lambda:full(x))); th=timing_summary(cuda_times(lambda:full(x,head_mask))); tn=timing_summary(cuda_times(lambda:narrow(x)))
metrics={"full_parameters":count_params(full),"head_mask_parameters":count_params(full),"ffn_narrow_parameters":count_params(narrow),"head_mask_rmse":tensor_metrics(ref,hm)["rmse"],"ffn_narrow_rmse":tensor_metrics(ref,fn)["rmse"],"layer_skip_rmse":tensor_metrics(ref,skip)["rmse"],"full_median_ms":tf["median_ms"],"head_mask_median_ms":th["median_ms"],"ffn_narrow_median_ms":tn["median_ms"],"sequence_length":128,"hidden_width":64,"full_ffn_width":256,"narrow_ffn_width":128}
analysis=(f"Masking two of four heads preserved {metrics['head_mask_parameters']:,} parameters and measured "
          f"{th['median_ms']:.6f} ms versus {tf['median_ms']:.6f} ms for the full block. Physical FFN narrowing "
          f"reduced parameters to {metrics['ffn_narrow_parameters']:,}, measured {tn['median_ms']:.6f} ms, and introduced "
          f"RMSE {metrics['ffn_narrow_rmse']:.6f}. Layer skipping had RMSE {metrics['layer_skip_rmse']:.6f} before recovery.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Full parameters | 49,984 |
| FFN-narrow parameters | 33,472 |
| Head-mask RMSE | 0.037988 |
| FFN-narrow RMSE | 0.146989 |
| Layer-skip RMSE | 0.227471 |
| Full median | 0.153360 ms |
| FFN-narrow median | 0.211440 ms |


## 7. Interpret rather than merely print

Masking two of four heads preserved 49,984 parameters and measured 0.159792 ms versus 0.153360 ms for the full block. Physical FFN narrowing reduced parameters to 33,472, measured 0.211440 ms, and introduced RMSE 0.146989. Layer skipping had RMSE 0.227471 before recovery.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 22,
    "title": 'Pruning Transformer Heads, FFN Neurons, and Layers',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Transformer pruning must name the structural unit and prove its physical compute path; masks alone are insufficient.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 22,
  "title": "Pruning Transformer Heads, FFN Neurons, and Layers",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260830
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "full_parameters": 49984,
    "head_mask_parameters": 49984,
    "ffn_narrow_parameters": 33472,
    "head_mask_rmse": 0.037987977266311646,
    "ffn_narrow_rmse": 0.14698943495750427,
    "layer_skip_rmse": 0.2274712175130844,
    "full_median_ms": 0.1533600017428398,
    "head_mask_median_ms": 0.159791998565197,
    "ffn_narrow_median_ms": 0.21143999695777893,
    "sequence_length": 128,
    "hidden_width": 64,
    "full_ffn_width": 256,
    "narrow_ffn_width": 128
  },
  "analysis": "Masking two of four heads preserved 49,984 parameters and measured 0.159792 ms versus 0.153360 ms for the full block. Physical FFN narrowing reduced parameters to 33,472, measure

## 9. Make the bounded decision

> Transformer pruning must name the structural unit and prove its physical compute path; masks alone are insufficient.

**Acceptance/rollback:** Accept a Transformer structure only after task/perplexity gates and a runtime trace confirm that the intended dimensions or depth changed.

**Failure analysis:** Random weights do not reveal head redundancy. Fused attention kernels may require fixed head dimensions or grouped-query layouts, and KV-cache shape couples attention structure to serving memory. Layer removal can shift normalization statistics and generation behavior.


## 10. Extend the evidence

Repeat on a pretrained encoder or decoder, evaluate task quality/perplexity and KV-cache bytes, and use a backend with explicit variable-head or narrowed-FFN support.

The full evidence boundary and references are in [`README.md`](README.md).
